# 03. First pass on real data

**Scope.** Running `integral_distance` and `p_variation_exact` on archive data, and reporting what comes back.

Exploratory. Output is a description of behaviour, with no claim that one discrepancy beats another. Comparison against MSE as a similarity measure is designed separately in `docs/protocol_01_similarity.md`, held until implementations are known to behave.

Order matters: a comparison run on unvalidated code produces numbers with no attribution. So behaviour first.

**Contents.** §1 data and constructed time axis. §2 quantities computed. §3 metric axioms. §4 magnitude and spread. §5 cost. §6 agreement between MSE and $V_p$. §7 limits.

---

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from aeon.datasets import load_classification

import sys; sys.path.insert(0, "../src")
from pathloss.norms import integral_distance, integral_norm, pointwise_mse
from pathloss.pvar import p_variation_exact, p_variation_pruned

rng = np.random.default_rng(0)
DATASET = "BasicMotions"      # small: 40 train, 6 channels, 100 points
EXTRACT = "../data/raw"

## 1. Data, and a time axis we construct

Source: Multiverse archive (Middlehurst et al., 2026, [arXiv:2603.20352](https://arxiv.org/abs/2603.20352)), loaded through `aeon`. `MV-core` is a 66-dataset subset recommended for initial work, listed as `aeon.datasets.tsc_datasets.multiverse_core`.

`load_classification` returns shape $(n_{\text{cases}}, d, N)$ for $d$ channels and $N$ time points. `pathloss` takes $(\dots, N, d)$, so cases are transposed on load.

### 1.1 Absence of a time axis

Dataset metadata reports `timestamps: False`. Archive supplies ordered values and no times. So

$$t_i \;=\; \frac{i}{N-1}, \qquad i = 0, \dots, N-1$$

is a construction rather than a measurement, and every $L^p$ quantity below is taken against it. Two consequences.

Uniform sampling (13/08) asserts nothing about this data, there being no sampling times to be uniform. Under $t_i = i/(N-1)$, trapezoid weights and MSE weights differ only in two endpoint half-weights, so `integral_distance(..., p=2, normalise=True)**2` and `pointwise_mse` agree to $O(1/N)$ by construction. §6 uses that as a check on the code.

$V_p$ ignores $t$ entirely, being a supremum over subsequences of values, so it is unaffected by the choice.

In [ ]:
Xtr, ytr = load_classification(DATASET, split="train", extract_path=EXTRACT)
Xte, yte = load_classification(DATASET, split="test", extract_path=EXTRACT)

paths = np.transpose(Xtr, (0, 2, 1))          # (n_cases, N, d)
n_cases, N, d = paths.shape
t = np.linspace(0.0, 1.0, N)

print(f"{DATASET}: {n_cases} train cases, {d} channels, {N} points")
print("classes:", dict(zip(*np.unique(ytr, return_counts=True))))
print("finite:", np.isfinite(paths).all())
print("per-channel mean/std of case 0:", paths[0].mean(0).round(3), paths[0].std(0).round(3))

## 2. Quantities computed

For sampled paths $x, y : \{t_0, \dots, t_{N-1}\} \to \mathbb{R}^d$, with $|\cdot|$ Euclidean on $\mathbb{R}^d$ and $w_i = \tfrac12(\Delta t_{i-1} + \Delta t_i)$ trapezoid weights (notebook 01 §1):

$$\mathrm{MSE}(x,y) = \frac1N \sum_i |x_i - y_i|^2 ,
\qquad
\|x - y\|_{L^p} = \Big(\sum_i w_i\,|x_i - y_i|^p\Big)^{1/p} ,$$

$$V_p(x - y) = \Big(\sup \sum_k \big|(x-y)_{n_k} - (x-y)_{n_{k-1}}\big|^p\Big)^{1/p},$$

supremum over increasing subsequences $0 = n_0 < \cdots < n_K = N-1$ (notebook 02 §1).

$V_p$ is applied to difference path $x - y$ rather than as a gap $|V_p(x) - V_p(y)|$. Subadditivity of $V_p$ gives $|V_p(x) - V_p(y)| \le V_p(x-y)$, so the gap is a lower bound vanishing on every pair of equal roughness (`docs/open_questions.md` Q3, Q20).

$V_p(x-y) + |x_0 - y_0|$ is a metric, $V_p$ being a seminorm vanishing exactly on constants (Q5). Offset term is dropped below, all quantities being reported for comparison rather than as certified distances.

In [ ]:
def discrepancies(a, b, ps=(1.0, 2.0, 3.0)):
    """All quantities for one pair of paths, each shape (N, d)."""
    out = {"mse": float(pointwise_mse(a, b))}
    for p in ps:
        out[f"L{p:g}"] = float(integral_distance(t, a, b, p=p, normalise=True))
        out[f"V{p:g}"] = float(p_variation_exact(a - b, p=p))
    out["Linf"] = float(integral_distance(t, a, b, p=np.inf))
    return out

discrepancies(paths[0], paths[1])

## 3. Metric axioms

Checked on the data rather than in theory, as a test of implementation. For $d(\cdot,\cdot)$ each quantity of §2 and paths $a, b, c$ drawn from the training set:

$$d(a,a) = 0, \qquad d(a,b) = d(b,a), \qquad d(a,c) \le d(a,b) + d(b,c) .$$

Identity of indiscernibles is untestable here, distinct cases being distinct by construction.

Failure of any of these indicates a bug rather than a property of the data, every quantity in §2 being a norm of the difference path.

In [ ]:
def axiom_report(fn_name, fn, trials=200, tol=1e-9):
    zero = max(abs(fn(a, a)) for a in paths)
    sym, tri = 0.0, 0
    for _ in range(trials):
        i, j, k = rng.choice(n_cases, 3, replace=False)
        a, b, c = paths[i], paths[j], paths[k]
        sym = max(sym, abs(fn(a, b) - fn(b, a)))
        if fn(a, c) > fn(a, b) + fn(b, c) + tol:
            tri += 1
    return {"self": zero, "asym": sym, "triangle_violations": tri}

fns = {
    "mse":  lambda a, b: float(pointwise_mse(a, b)),
    "L2":   lambda a, b: float(integral_distance(t, a, b, p=2.0, normalise=True)),
    "V2":   lambda a, b: float(p_variation_exact(a - b, p=2.0)),
}
for name, fn in fns.items():
    print(name, axiom_report(name, fn))

## 4. Magnitude and spread

Two properties bearing on use as a training objective.

**Magnitude** fixes gradient scale. A quantity returning $10^{-8}$ or $10^{12}$ needs rescaling before it is combined with any other term.

**Spread** decides whether the quantity carries information at all. A discrepancy returning nearly the same value on every pair separates nothing, whatever its theoretical properties. Reported as coefficient of variation $\sigma/\mu$ over sampled pairs, dimensionless and so comparable across quantities of different units.

In [ ]:
m = 300
idx = rng.choice(n_cases, size=(m, 2))
idx = idx[idx[:, 0] != idx[:, 1]]

rows = [discrepancies(paths[i], paths[j]) for i, j in idx]
keys = rows[0].keys()
vals = {k: np.array([r[k] for r in rows]) for k in keys}

print(f"{'quantity':8} {'min':>12} {'median':>12} {'max':>12} {'cv':>8}")
for k, v in vals.items():
    print(f"{k:8} {v.min():12.4g} {np.median(v):12.4g} {v.max():12.4g} {v.std()/v.mean():8.3f}")

## 5. Cost

`p_variation_exact` is $O(N^2)$ with $N$ sequential steps, each vectorised over candidate index (notebook 02 §3). `integral_distance` is $O(N)$.

Two figures matter for a training loop: time per pair at the $N$ of this dataset, and growth with $N$. A backward pass adds little given Danskin's theorem, gradient of $V_p$ following the maximising subsequence already recorded by the forward pass (Q3). Cost inside a loop is Q25.

MV-core reaches $N = 17{,}984$, so growth is measured rather than assumed.

In [ ]:
def timeit(fn, reps=20):
    s = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - s) / reps

a, b = paths[0], paths[1]
print(f"N = {N}")
print(f"  integral_distance  {timeit(lambda: integral_distance(t, a, b, p=2.0))*1e6:8.1f} us")
print(f"  p_variation_exact  {timeit(lambda: p_variation_exact(a - b, p=2.0))*1e6:8.1f} us")
print(f"  p_variation_pruned {timeit(lambda: p_variation_pruned(a - b, p=2.0))*1e6:8.1f} us")

for n in (100, 200, 400, 800, 1600):
    z = rng.normal(size=(n, d)).cumsum(0)
    print(f"N = {n:5d}  exact {timeit(lambda: p_variation_exact(z, p=2.0), reps=5)*1e3:7.2f} ms")

## 6. Agreement between MSE and $V_p$

MSE and $L^2$ agree to $O(1/N)$ under §1.1's constructed grid, so their ratio is a check on the code: departure from 1 beyond $O(1/N)$ indicates an error in weights.

$V_2(x-y)$ against MSE is a description. Rank correlation near 1 means the two order pairs alike on this data, and $V_p$ then adds nothing here; correlation well below 1 means they disagree about which pairs are close. Which ordering is preferable is undetermined by this plot, no ground truth for path similarity being available.

Spearman rank correlation $\rho_S$ is used rather than Pearson, quantities differing in units and monotone relation being what matters.

In [ ]:
ratio = vals["L2"] ** 2 / vals["mse"]
print(f"L2^2 / MSE: median {np.median(ratio):.6f}, expected 1 + O(1/N) with N = {N}")

rho, _ = spearmanr(vals["mse"], vals["V2"])
print(f"Spearman(MSE, V2) = {rho:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(vals["mse"], vals["L2"] ** 2, s=8)
ax[0].set(xlabel="MSE", ylabel=r"$\|x-y\|_{L^2}^2$", title="agreement check")
ax[1].scatter(vals["mse"], vals["V2"], s=8)
ax[1].set(xlabel="MSE", ylabel=r"$V_2(x-y)$", title=fr"$\rho_S$ = {rho:.3f}")
for a_ in ax:
    a_.set_xscale("log"); a_.set_yscale("log")
fig.tight_layout()

## 7. Limits of this pass

Established here: implementations run on archive data, return finite numbers, satisfy metric axioms on sampled triples, and cost what §5 measures.

Undetermined here, and each requiring its own design:

- Which ordering of pairs is preferable. §6 shows agreement or disagreement, with no ground truth against which either is right. `docs/protocol_01_similarity.md` is the design that supplies one.
- Whether any quantity trains a model. Differentiability is settled in principle (Q3), and `src/pathloss/losses.py` is unwritten.
- Behaviour on one dataset generalising to MV-core. One dataset is a smoke test.
- Choice of $p$. Values 1, 2, 3 are reported side by side. Index $p^\ast = \inf\{p : V_p < \infty\}$ is unimplemented (Q6), so no value here is principled.

## Next

1. Repeat §§3 to 5 across MV-core, recording failures and costs per dataset.
2. Implement $p^\ast$ estimation, validating on fractional Brownian motion where $p^\ast = 1/H$ (Q6).
3. Port `integral_distance` and $V_p$ to differentiable torch losses, tested against these NumPy values.

## References

- **Middlehurst, Rushbrooke, Ismail-Fawaz, Devanne, Forestier, Dempster, Webb, Holder & Bagnall (2026)**, *The Multiverse of Time Series Machine Learning*, [arXiv:2603.20352](https://arxiv.org/abs/2603.20352). §1.
- Definitions and their verification: notebook 01 §1 (weights), notebook 02 §§1, 3 ($p$-variation), `tests/`.
- Open items referenced as Q$n$: `docs/open_questions.md`.